# Triển khai "Bản đồ lợi nhuận ròng"

Đọc các dữ liệu

In [24]:
import pandas as pd
import numpy as np


# --- BƯỚC 1: CẤU HÌNH & NẠP DỮ LIỆU ---
path = '../../data/'
print("🚀 Đang khởi động Pipeline xử lý dữ liệu...")

# Hợp nhất các file bị chia nhỏ (Dùng glob để chống lỗi thiếu file)
df_orders = pd.read_csv(path + 'orders.csv')
df_items = pd.read_csv(path + 'order_items.csv')
df_products = pd.read_csv(path + 'products.csv')
df_shipments = pd.read_csv(path + 'shipments.csv')
df_returns = pd.read_csv(path + 'returns.csv')
df_customers = pd.read_csv(path + 'customers.csv')
df_geo = pd.read_csv(path + 'geography.csv')

🚀 Đang khởi động Pipeline xử lý dữ liệu...


/var/folders/n6/fpv8j7h17fxb5lbxwtz4qbyr0000gn/T/ipykernel_62087/1490183271.py:11: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(path + 'order_items.csv')


TÍNH TOÁN LỢI NHUẬN GỘP (GROSS PROFIT)

In [25]:
# Lấy giá vốn (cogs) từ bảng products
df_temp = pd.merge(df_items, df_products[['product_id', 'cogs']], on='product_id', how='left')

# Công thức: Gross Profit = (Giá bán - Giá vốn) * Số lượng
df_temp['item_gross_profit'] = (df_temp['unit_price'] - df_temp['cogs']) * df_temp['quantity']

# Tổng hợp theo từng đơn hàng (order_id)
df_order_base = df_temp.groupby('order_id').agg({
    'item_gross_profit': 'sum',
    'unit_price': 'sum'  # Doanh thu gốc của đơn hàng
}).reset_index()

XỬ LÝ LOGIC LỢI NHUẬN RÒNG & SHIP/RETURN: 
Dùng Vectorization (np.where) để tính Net Profit cực nhanh
- Nếu hoàn: Lỗ = -(Phí ship + Phí xử lý 20k)
- Nếu thành công: Lãi = Lợi nhuận gộp - Phí ship

In [26]:
# Gộp phí ship và dữ liệu trả hàng
df_logic = pd.merge(df_order_base, df_shipments[['order_id', 'shipping_fee']], on='order_id', how='left')
df_logic = pd.merge(df_logic, df_returns[['order_id', 'return_date']], on='order_id', how='left')

# Xử lý các giá trị Null (để không bị lỗi khi tính toán)
df_logic['shipping_fee'] = df_logic['shipping_fee'].fillna(0)
df_logic['is_returned'] = df_logic['return_date'].notnull().astype(int)

df_logic['net_profit'] = np.where(
    df_logic['is_returned'] == 1,
    -(df_logic['shipping_fee'] + 20000),
    df_logic['item_gross_profit'] - df_logic['shipping_fee']
)

Gắn yếu tố địa lý

In [27]:
# Làm sạch bảng tra cứu để tránh "phình" dữ liệu (Data Explosion)
cust_map = df_customers[['customer_id', 'city']].drop_duplicates('customer_id')
geo_map = df_geo[['city', 'region']].drop_duplicates('city')

# Gộp bắc cầu: Logic -> Orders -> Customers -> Geography
df_final = df_logic.merge(
    df_orders[['order_id', 'customer_id']], on='order_id', how='left'
).merge(
    cust_map, on='customer_id', how='left'
).merge(
    geo_map, on='city', how='left'
)

df_final.to_csv('geography_profit_ready.csv', index=False)